# Tarea 2 — Reconstrucción 3D con COLMAP
**IIC3912 Tópicos Avanzados de Gráfica Computacional**

Este notebook es el punto de partida común para todos los experimentos.
Ejecuta las celdas en orden para instalar dependencias, montar Drive y descargar el dataset.

In [1]:
# Instalación de dependencias
!pip install pycolmap plotly open3d pandas tqdm -q

import os, shutil, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pycolmap
from pathlib import Path
from PIL import Image as PILImage

print('pycolmap:', pycolmap.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 835.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.3 MB/s eta 0:00:00
pycolmap: 4.0.4


In [2]:
SCENES = ['garden', 'bicycle', 'bonsai', 'counter']

IMG_SCALE = 4  # resolución: 1 (full), 2, 4, 8  — se recomienda 4 en Colab CPU

In [3]:
# ── Google Drive y paths ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/tarea2_mipnerf')
BASE       = Path('/content/colmap_work')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

def scene_paths(scene):
    return {
        'drive'   : DRIVE_BASE / scene,
        'images'  : BASE / scene / 'images',
        'sparse'  : BASE / scene / 'sparse',
        'db'      : BASE / scene / 'colmap.db',
    }

# Crear directorios de trabajo para todas las escenas
for scene in SCENES:
    p = scene_paths(scene)
    for d in [p['images'], p['sparse'] / 'student', p['sparse'] / 'official']:
        d.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
# ── Descarga y extracción del dataset ─────────────────────────────────────
# El zip (~1.8 GB) se descarga una sola vez a Drive.
# Las sesiones siguientes lo reutilizan directamente.

DATASET_URL = 'https://storage.googleapis.com/gresearch/refraw360/360_v2.zip'
ZIP_PATH    = DRIVE_BASE / '360_v2.zip'

def download_and_extract(scene):
    dst = DRIVE_BASE / scene
    if dst.exists():
        print(f'[{scene}] ya en Drive')
        return
    if not ZIP_PATH.exists():
        print('Descargando dataset (~1.8 GB)...')
        ret = os.system(f'wget -q --show-progress -O "{ZIP_PATH}" "{DATASET_URL}"')
        if ret != 0:
            raise RuntimeError('Descarga fallida. Sube el zip manualmente a Drive.')
    print(f'[{scene}] extrayendo...')
    ret = os.system(f'unzip -q "{ZIP_PATH}" "{scene}/*" -d "{DRIVE_BASE}"')
    if ret != 0 or not dst.exists():
        raise RuntimeError(f'Error extrayendo {scene}')
    print(f'[{scene}] listo en {dst}')

def prepare_images(scene):
    paths = scene_paths(scene)
    dst   = paths['images']
    existing = list(dst.glob('*.jpg')) + list(dst.glob('*.JPG')) + list(dst.glob('*.png'))
    if existing:
        print(f'[{scene}] {len(existing)} imágenes ya en local')
        return
    scale_folder = f'images_{IMG_SCALE}' if IMG_SCALE > 1 else 'images'
    src = paths['drive'] / scale_folder
    if not src.exists():
        src = paths['drive'] / 'images'
        print(f'[{scene}] {scale_folder}/ no encontrada → usando images/')
    imgs = sorted(list(src.glob('*.jpg')) + list(src.glob('*.JPG')) + list(src.glob('*.png')))
    print(f'[{scene}] copiando {len(imgs)} imágenes desde {src.name}...')
    for img_path in imgs:
        shutil.copy(str(img_path), str(dst / img_path.name))
    with PILImage.open(list(dst.glob('*'))[0]) as im:
        print(f'[{scene}] resolución: {im.size} | total: {len(imgs)} imgs')

# Descargar y preparar las 4 escenas
for scene in SCENES:
    download_and_extract(scene)
    prepare_images(scene)

[garden] ya en Drive ✓
[garden] copiando 185 imágenes desde images_4...
[garden] resolución: (1297, 840) | total: 185 imgs ✓
[bicycle] extrayendo...


# Analisis visual y semántico

Tenemos 2 escenas a comparar, contamos con garden y counter, una de exterior y otra de interior, y curiosamente, según nuestro analisis hecho en el experimento B, una de buena calidad y robusta y la otra de menor calidad y menos robusta.
Compararemos la reconstrucción sparse que hicimos, con la densa (Multi-View Stereo) y la malla superficial continua (Poisson Surface Reconstruccion), además de contrastar entre escenas.

Tomamos una captura de cada reconstrucción desde un mismo ángulo, para hacer un análisis más preciso. Cabe mencionar que entre los modelos entregados y el reconstruido propio diferían en el sistema de coordenadas, debido a la reorientación hecha en el experimento D.

Es posible observar la relación entre SfM y MVS. Como mencionabamos anteriormente, SfM entrega una nube de puntos, es una representación esquelética. Estos puntos que son vistos desde distintas camaras, zonas donde la extracción y el featuring matching es existoso, y son zonas en donde existe un alto gradiente de texturas. Por otro lado, tenemos la nube densa entregada por Multi-View Stereo (MVS), la cual sabemos toma las poses estimadas por SfM y procede a inferir la profundidad de cada pixel. Con esto se logra rellenar las superficies que carecen de características distintivas, texturas, tal como es visible al observar, por ejemplo, el pasto del jardín de la escena garden o la superficie del mesón de la cocina de la escena counter.

Luego al observar la malla de Poisson se podría considerar la final, es visible el funcionamiento del algoritmo de Poisson Surface Reconstruction. Como vimos en clases, este método no conecta puntos, sino que utiliza estos puntos y sus respectivas normales orientadas como restricción para encontrar una función que permita para reconstruir una superficie global, suave y coherente. Es posible observar en las mallas de cada escena unos polígonos gigantes distorsionados, que corresponderían a partes de la perisferia de las escenas, tales como los fondos o "techos" de las imagenes. Con esto vemos la naturaleza watertight del algoritmo, buscando sellar la escena completa incluso cuando no hay información suficiente entre "puntos"/"espacios".

### Comparación escenas

 - Garden (outdoor): Se observa, dada la naturaleza misma de un jardín, con harta vegetación, piedritas, diferentes profundidades al ser espacio abierto, que es muy rica en texturas y sin superficies reflexantes, sino más bien mate, permite lograr una buena consistencia entre las diferentes vistas. Lo anterior permitió que las normales de MVS fueran bastante precisas. En el caso de Poisson, generó una superficie de muy buena calidad, donde el enfoque global y suave permitió que la escena sea coherente y limpia.

 - Counter (indoor): En este caso, como habíamos previsto y determinando anteriormente, vemos los puntos bajos del pipeline. Esta escena tiene características que en clases hemos hablado, generan problemas para generar una buena reconstrucción, tales como superficies reflectantes, plásticos de empaque y materiales de metal, tal como el lavaplato. Debido a que el brillo cambia de posición según desde el ángulo que se toma la foto, el ángulo de la cámara, la triangulación de puntos no puede ser consistente, ni la profundidad de cada pixel, y menos de su normal. Por lo que genera ruido en MVS y, como mencionabamos, sus normales estan mal orientadas. Dada la característica de Poisson de forzar una función continua y suave, en esta escena con este algoritmo vemos el efecto de cera derretida mucho más marcado, fusionando partes o bordes no definidos.

